#### This notebook explains how to uncover risks related to your usecase based on a given taxonomy.


##### Import libraries


In [1]:
from typing import Any, Dict, List, Optional

from risk_atlas_nexus.data import load_resource
import json
from pathlib import Path
from jinja2 import Template
from risk_atlas_nexus.blocks.prompt_response_schema import LIST_OF_STR_SCHEMA

from risk_atlas_nexus.ai_risk_ontology.datamodel.ai_risk_ontology import Risk
from risk_atlas_nexus.blocks.inference import (
    OllamaInferenceEngine,
    RITSInferenceEngine,
    VLLMInferenceEngine,
    WMLInferenceEngine,
)
from risk_atlas_nexus.blocks.inference.params import (
    InferenceEngineCredentials,
    OllamaInferenceEngineParams,
    RITSInferenceEngineParams,
    VLLMInferenceEngineParams,
    WMLInferenceEngineParams,
)
from risk_atlas_nexus.library import RiskAtlasNexus

/Users/dhaval/Projects/Usage-Governance/risk-atlas-nexus/src/risk_atlas_nexus/toolkit/job_utils.py:2: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


##### Risk Atlas Nexus uses Large Language Models (LLMs) to infer risks dimensions. Therefore requires access to LLMs to inference or call the model.

**Available Inference Engines**: WML, Ollama, vLLM, RITS. Please follow the [Inference APIs](https://github.com/IBM/risk-atlas-nexus?tab=readme-ov-file#install-for-inference-apis) guide before going ahead.

_Note:_ RITS is intended solely for internal IBM use and requires TUNNELALL VPN for access.


In [2]:
inference_engine = OllamaInferenceEngine(
    model_name_or_path="granite3.2:8b",
    credentials=InferenceEngineCredentials(api_url="http://localhost:11434"),
    parameters=OllamaInferenceEngineParams(
        num_predict=1000, num_ctx=8192, temperature=0, repeat_penalty=1
    ),
)

# inference_engine = WMLInferenceEngine(
#     model_name_or_path="ibm/granite-20b-code-instruct",
#     credentials={
#         "api_key": "WML_API_KEY",
#         "api_url": "WML_API_URL",
#         "project_id": "WML_PROJECT_ID",
#     },
#     parameters=WMLInferenceEngineParams(
#         max_new_tokens=1000, decoding_method="greedy", repetition_penalty=1
#     ),
# )

# inference_engine = VLLMInferenceEngine(
#     model_name_or_path="ibm-granite/granite-3.1-8b-instruct",
#     credentials=InferenceEngineCredentials(
#         api_url="VLLM_API_URL", api_key="VLLM_API_KEY"
#     ),
#     parameters=VLLMInferenceEngineParams(max_tokens=1000, temperature=0.7),
# )

# inference_engine = RITSInferenceEngine(
#     model_name_or_path="ibm/granite-20b-code-instruct",
#     credentials={
#         "api_key": "RITS_API_KEY",
#         "api_url": "RITS_API_URL",
#     },
#     parameters=RITSInferenceEngineParams(max_tokens=1000, temperature=0.7),
# )

[2025-05-23 11:47:38:487] - INFO - RiskAtlasNexus - OLLAMA inference engine will execute requests on the server at http://localhost:11434.
[2025-05-23 11:47:38:530] - INFO - RiskAtlasNexus - Created OLLAMA inference engine.


##### Create an instance of RiskAtlasNexus

_Note: (Optional)_ You can specify your own directory in `RiskAtlasNexus(base_dir=<PATH>)` to utilize custom AI ontologies. If left blank, the system will use the provided AI ontologies.


In [3]:
risk_atlas_nexus = RiskAtlasNexus()

[2025-05-23 11:47:38:601] - INFO - RiskAtlasNexus - Created RiskAtlasNexus instance. Base_dir: None


#### Risk Generation Template


In [4]:
RISKS_GENERATION_TEMPLATE = """
        You are are an expert at AI risk classification. I want you to play the role of a risk compliance officer.
        Identify the Risks based on the usecase, question and the answer. Return a list of Risk categories in a json format.
        You should answer followed by an explanation on how that answer was generated. If answer doesn't fit into any of the above categories, classify it as: Unknown.

        Study the JSON below containing list of risk categories and their descriptions. 

        {{ risks }}

{% for example in examples %}
        Usecase: {{ example.intent }}
        Question: {{ example.question }}
        Answer: {{ example.answer }}
        Risks: {{ example.risks }}
{% endfor %}
        Usecase: {{ usecase }}
        Question: {{ question }}
        Answer: {{ answer }}
"""

#### User Intent


In [5]:
usecase = "Generate personalized, relevant responses, recommendations, and summaries of claims for customers to support agents to enhance their interactions with customers."

#### Load CoT data


In [6]:
questionnaire_cot_data = load_resource("risk_questionnaire_cot.json")
risk_cot_data = load_resource("risk_generation_cot.json")

#### Get Questionnaire Predictions and Risk List


In [7]:
questionnaire_preds: List[str] = (
    risk_atlas_nexus.generate_few_shot_risk_questionnaire_output(
        usecase=usecase,
        cot_data=questionnaire_cot_data,
        inference_engine=inference_engine,
    )
)

questionnaire_output = []
for questionnaire_data, questionnaire_pred in zip(
    questionnaire_cot_data, questionnaire_preds
):
    questionnaire_output.append(
        {"question": questionnaire_data["question"], "answer": questionnaire_pred}
    )

risks: List[Risk] = risk_atlas_nexus.get_all_risks(taxonomy="ibm-risk-atlas")

Inferring with OLLAMA: 100%|██████████| 7/7 [00:36<00:00,  5.20s/it]


In [8]:
questionnaire_output

[{'question': 'What domain does your use request fall under? Customer service/support, Technical, Information retrieval, Strategy, Code/software engineering, Communications, IT/business automation, Writing assistant, Financial, Talent and Organization including HR, Product, Marketing, Cybersecurity, Healthcare, User Research, Sales, Risk and Compliance, Design, Other',
  'answer': 'Customer service/support'},
 {'question': 'In which environment is the system used?',
  'answer': 'Customer Service or Claims Support Departments'},
 {'question': 'What techniques are utilised in the system? Multi-modal: {Document Question/Answering, Image and text to text, Video and text to text, visual question answering}, Natural language processing: {feature extraction, fill mask, question answering, sentence similarity, summarization, table question answering, text classification, text generation, token classification, translation, zero shot classification}, computer vision: {image classification, image

#### Prepare Prompts


In [11]:
prompts = []
for questionnaire in questionnaire_output:
    prompts.append(
        Template(RISKS_GENERATION_TEMPLATE).render(
            usecase=usecase,
            question=questionnaire["question"],
            answer=questionnaire["answer"],
            examples=risk_cot_data["examples"],
            risks=json.dumps(
                [
                    {"category": risk.name, "description": risk.description}
                    for risk in risks
                    if risk.name
                ],
                indent=2,
            ),
        )
    )

#### Infer risks


In [13]:
LIST_OF_STR_SCHEMA["items"]["enum"] = [risk.name for risk in risks]
inference_response = inference_engine.chat(
    prompts,
    response_format={
        "type": "object",
        "properties": {
            "answer": LIST_OF_STR_SCHEMA,
            "explanation": {"type": "string"},
        },
        "required": ["answer", "explanation"],
    },
    postprocessors=["json_object"],
    verbose=False,
)

identified_risks = []
for response in inference_response:
    identified_risks.extend(response.prediction["answer"])

risks = list(set(identified_risks))

print(risks)

['Unrepresentative data', 'Poor model accuracy', 'Data privacy rights alignment', 'Lack of model transparency', 'Exposing personal information', 'Incomplete usage definition', 'Output bias', 'Revealing confidential information', 'Hallucination', 'Improper usage']
